# Bronze -> Silver: source-agnostic health & training model

Builds Strava's slice of the conformed Silver model from the raw payloads in Bronze schema `strava`. Every Silver table
carries `source_system`; this notebook only replaces rows where `source_system = 'strava'`, so Garmin, Apple Health,
Oura, WHOOP, manual entry and the iOS app can write into the same tables with their own transform notebooks.
Shared reference data (DataSource, ActivityTypeMapping, HealthMetric) and contract tables come from `Silver_Reference_Data`.

Conventions
- One row per real-world entity, identified by a deterministic hash key (`*_id`) plus `source_system` + `source_*_id` lineage.
- Canonical SI units in column names (`_m`, `_s`, `_mps`, `_bpm`, `_w`, `_kg`, `_kcal`, `_c`); presentation units belong in Gold.
- UTC timestamps (`*_utc`) plus wall-clock local time (`*_local`) and `local_date` for day attribution.
- Every source writes only its own slice (`replaceWhere source_system = '<source>'`), so sources never overwrite each other.
- Source-specific fields that do not fit the canonical model are kept in `source_attributes_json`.
- Contract tables (sleep, nutrition, goals) are created empty by `Silver_Reference_Data` so new sources plug in without schema changes.

In [ ]:
rebuild_tables = False

In [ ]:
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import json

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.ansi.enabled", "false")
spark.conf.set("spark.sql.parquet.vorder.default", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")

SOURCE_SYSTEM = "strava"
WORKSPACE_ID = notebookutils.runtime.context["currentWorkspaceId"]
def lakehouse_id(name):
    """Resolve a lakehouse in this workspace by name, so the notebook works in any workspace."""
    try:
        item = notebookutils.lakehouse.get(name)
        return item["id"] if isinstance(item, dict) else item.id
    except Exception:
        import sempy.fabric as fabric
        return fabric.resolve_item_id(name, type="Lakehouse")


BRONZE_LAKEHOUSE_ID = lakehouse_id("Bronze")   # Bronze lakehouse (multi-source, schema per source system)
BRONZE_TABLES = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{BRONZE_LAKEHOUSE_ID}/Tables/{SOURCE_SYSTEM}"
PROCESSED_AT = datetime.now(timezone.utc).replace(microsecond=0)
SOURCE_SCOPE = f"source_system = '{SOURCE_SYSTEM}'"
row_counts = {}


def read_bronze(table_name):
    path = f"{BRONZE_TABLES}/{table_name}"
    return spark.read.format("delta").load(path) if notebookutils.fs.exists(path) else None


def hash_key(*cols):
    parts = [F.coalesce(c.cast("string"), F.lit("")) for c in cols]
    return F.substring(F.sha2(F.concat_ws("|", *parts), 256), 1, 32)


def strava_user_id(athlete_id_col):
    return F.concat(F.lit("usr_"), F.substring(F.sha2(F.concat(F.lit("strava:"), athlete_id_col.cast("string")), 256), 1, 24))


def latest(df, keys):
    w = Window.partitionBy(*keys).orderBy(F.desc("ingested_at"), F.desc("batch_id"))
    return df.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn")


def table_exists(name):
    return notebookutils.fs.exists(f"Tables/{name}/_delta_log")


def read_table(name):
    return spark.read.format("delta").load(f"Tables/{name}")


def write_table(df, name, scope=SOURCE_SCOPE):
    """Overwrite only this source's slice of a Silver table (full overwrite for reference tables or rebuilds).

    Tables are written by path so the PascalCase table name is preserved in the lakehouse."""
    df = df.withColumn("silver_processed_at", F.lit(PROCESSED_AT).cast("timestamp"))
    writer = df.write.format("delta").mode("overwrite")
    if scope is None or rebuild_tables or not table_exists(name):
        writer.option("overwriteSchema", "true").save(f"Tables/{name}")
    else:
        writer.option("replaceWhere", scope).option("mergeSchema", "true").save(f"Tables/{name}")
    row_counts[name] = read_table(name).count()
    print(f"{name}: {row_counts[name]} rows")


def create_contract_table(name, ddl):
    if not table_exists(name):
        spark.createDataFrame([], ddl).write.format("delta").save(f"Tables/{name}")
    row_counts[name] = read_table(name).count()
    print(f"{name}: contract table ready ({row_counts[name]} rows)")

In [ ]:
# Shared reference data is maintained by the Silver_Reference_Data notebook (runs before every source transform)
ref_activity_type = read_table("ActivityTypeMapping").where(F.col("source_system") == SOURCE_SYSTEM).drop("silver_processed_at")
ref_data_source = read_table("DataSource")

In [ ]:
# Read Bronze (latest landed version of every source record)

# Bronze tables are named after the Strava API endpoint they come from
activities_raw = read_bronze("athlete_activities")              # GET /athlete/activities
if activities_raw is None:
    raise RuntimeError("Bronze table strava.athlete_activities not found. Run Strava_Ingest first.")

API_RAW_DDL = ("source_system string, source_entity string, source_record_id string, parent_record_id string, athlete_id string, "
               "api_endpoint string, http_status int, raw_payload string, payload_hash string, ingested_at timestamp, "
               "ingestion_date date, batch_id string")
api_raw = spark.createDataFrame([], API_RAW_DDL)
for bronze_table in ["activities", "activities_streams", "athlete", "athlete_zones"]:
    df = read_bronze(bronze_table)
    if df is not None:
        api_raw = api_raw.unionByName(df.select(*[c.split(" ")[0] for c in API_RAW_DDL.split(", ")]))

api_ok = api_raw.where("http_status = 200")
summary_raw = latest(activities_raw, ["source_record_id"])
detail_raw = latest(api_ok.where("source_entity = 'activity_detail'"), ["source_record_id"])
streams_raw = latest(api_ok.where("source_entity = 'activity_streams'"), ["source_record_id"])
athlete_raw = api_ok.where("source_entity = 'athlete'")
zones_raw = latest(api_ok.where("source_entity = 'athlete_zones'"), ["athlete_id"])

SUMMARY_SCHEMA = (
    "id long, name string, type string, sport_type string, workout_type int, start_date string, start_date_local string, "
    "timezone string, utc_offset double, distance double, moving_time long, elapsed_time long, total_elevation_gain double, "
    "elev_high double, elev_low double, average_speed double, max_speed double, average_cadence double, average_temp double, "
    "average_watts double, weighted_average_watts double, max_watts double, kilojoules double, device_watts boolean, "
    "has_heartrate boolean, average_heartrate double, max_heartrate double, suffer_score double, trainer boolean, "
    "commute boolean, manual boolean, private boolean, visibility string, gear_id string, external_id string, "
    "upload_id long, achievement_count int, kudos_count int, pr_count int, athlete struct<id:long>"
)
LAP_SCHEMA = (
    "struct<id:long, lap_index:int, name:string, elapsed_time:long, moving_time:long, start_date:string, distance:double, "
    "total_elevation_gain:double, average_speed:double, max_speed:double, average_cadence:double, average_watts:double, "
    "average_heartrate:double, max_heartrate:double, pace_zone:int>"
)
SPLIT_SCHEMA = (
    "struct<split:int, distance:double, elapsed_time:long, moving_time:long, elevation_difference:double, average_speed:double, "
    "average_grade_adjusted_speed:double, average_heartrate:double, pace_zone:int>"
)
BEST_EFFORT_SCHEMA = "struct<id:long, name:string, elapsed_time:long, moving_time:long, start_date:string, distance:double, pr_rank:int>"
DETAIL_SCHEMA = (
    f"calories double, description string, device_name string, perceived_exertion double, "
    f"gear struct<id:string, name:string, distance:double>, laps array<{LAP_SCHEMA}>, "
    f"splits_metric array<{SPLIT_SCHEMA}>, best_efforts array<{BEST_EFFORT_SCHEMA}>"
)

summary = summary_raw.select(
    "source_record_id", "athlete_id", "ingested_at", "batch_id", F.from_json("raw_payload", SUMMARY_SCHEMA).alias("a"))
detail = detail_raw.select(
    "source_record_id", F.col("ingested_at").alias("detail_ingested_at"), F.from_json("raw_payload", DETAIL_SCHEMA).alias("d"))

workouts_parsed = (
    summary.join(detail, "source_record_id", "left")
    .select(
        F.lit(SOURCE_SYSTEM).alias("source_system"),
        F.col("source_record_id").alias("source_workout_id"),
        F.coalesce(F.col("athlete_id"), F.col("a.athlete.id").cast("string")).alias("source_user_id"),
        F.col("a.external_id").alias("source_external_id"),
        F.col("a.sport_type").alias("source_activity_type"),
        F.col("a.type").alias("strava_type"),
        F.col("a.workout_type").alias("strava_workout_type"),
        F.col("a.name").alias("workout_name"),
        F.col("d.description").alias("workout_description"),
        F.to_timestamp("a.start_date").alias("start_time_utc"),
        F.to_timestamp("a.start_date_local").alias("start_time_local"),
        F.when(F.regexp_extract("a.timezone", r"\)\s*(.+)$", 1) != "", F.regexp_extract("a.timezone", r"\)\s*(.+)$", 1)).alias("timezone_name"),
        F.col("a.utc_offset").cast("int").alias("utc_offset_s"),
        F.col("a.elapsed_time").alias("elapsed_duration_s"),
        F.col("a.moving_time").alias("moving_duration_s"),
        F.col("a.distance").alias("distance_m"),
        F.col("a.total_elevation_gain").alias("elevation_gain_m"),
        F.col("a.elev_high").alias("elevation_high_m"),
        F.col("a.elev_low").alias("elevation_low_m"),
        F.col("a.average_speed").alias("avg_speed_mps"),
        F.col("a.max_speed").alias("max_speed_mps"),
        F.col("a.average_heartrate").alias("avg_heart_rate_bpm"),
        F.col("a.max_heartrate").alias("max_heart_rate_bpm"),
        F.col("a.average_watts").alias("avg_power_w"),
        F.col("a.max_watts").alias("max_power_w"),
        F.col("a.weighted_average_watts").alias("normalized_power_w"),
        F.col("a.device_watts").alias("is_power_from_device"),
        F.col("a.average_cadence").alias("avg_cadence"),
        F.col("a.average_temp").alias("avg_temperature_c"),
        F.col("a.kilojoules").alias("energy_kj"),
        F.col("d.calories").alias("active_energy_kcal"),
        F.col("d.perceived_exertion").alias("perceived_exertion"),
        F.col("a.suffer_score").alias("source_effort_score"),
        F.coalesce(F.col("a.trainer"), F.lit(False)).alias("is_trainer"),
        F.coalesce(F.col("a.commute"), F.lit(False)).alias("is_commute"),
        F.coalesce(F.col("a.manual"), F.lit(False)).alias("is_manual"),
        F.coalesce(F.col("a.private"), F.lit(False)).alias("is_private"),
        F.coalesce(F.col("a.has_heartrate"), F.lit(False)).alias("has_heart_rate"),
        F.col("d.device_name").alias("device_name"),
        F.col("a.gear_id").alias("source_equipment_id"),
        F.col("d.gear.name").alias("equipment_name"),
        F.col("detail_ingested_at").isNotNull().alias("has_detail"),
        F.to_json(F.struct(
            F.col("a.type").alias("strava_type"),
            F.col("a.workout_type").alias("strava_workout_type"),
            F.col("a.visibility").alias("visibility"),
            F.col("a.upload_id").alias("upload_id"),
            F.col("a.achievement_count").alias("achievement_count"),
            F.col("a.kudos_count").alias("kudos_count"),
            F.col("a.pr_count").alias("pr_count"),
        )).alias("source_attributes_json"),
        F.col("d.laps").alias("laps"),
        F.col("d.splits_metric").alias("splits_metric"),
        F.col("d.best_efforts").alias("best_efforts"),
        "ingested_at",
        "batch_id",
    )
    .join(F.broadcast(ref_activity_type), ["source_system", "source_activity_type"], "left")
)

dq_errors = F.array(
    F.when(F.col("source_workout_id").isNull(), F.lit("missing_source_id")),
    F.when(F.col("start_time_utc").isNull(), F.lit("missing_start_time")),
    F.when(F.coalesce(F.col("elapsed_duration_s"), F.lit(0)) <= 0, F.lit("non_positive_duration")),
    F.when(F.col("distance_m") < 0, F.lit("negative_distance")),
)
dq_warnings = F.array(
    F.when(~F.col("avg_heart_rate_bpm").between(25, 250), F.lit("implausible_avg_heart_rate")),
    F.when(~F.col("max_heart_rate_bpm").between(25, 250), F.lit("implausible_max_heart_rate")),
    F.when(F.col("avg_power_w") > 2000, F.lit("implausible_avg_power")),
    F.when(F.col("moving_duration_s") > F.col("elapsed_duration_s"), F.lit("moving_exceeds_elapsed")),
    F.when(F.col("is_distance_based") & ~F.coalesce(F.col("is_indoor_default"), F.lit(False)) & ~F.col("is_trainer")
           & (F.coalesce(F.col("distance_m"), F.lit(0.0)) == 0), F.lit("zero_distance_for_distance_sport")),
    F.when(F.col("activity_type").isNull(), F.lit("unmapped_activity_type")),
)

workouts_checked = (
    workouts_parsed
    .withColumn("dq_errors", F.filter(dq_errors, lambda x: x.isNotNull()))
    .withColumn("dq_warnings", F.filter(dq_warnings, lambda x: x.isNotNull()))
    .withColumn("user_id", strava_user_id(F.col("source_user_id")))
    .withColumn("workout_id", hash_key(F.col("source_system"), F.col("source_workout_id")))
)

workouts_valid = (
    workouts_checked
    .where(F.size("dq_errors") == 0)
    .withColumn("avg_heart_rate_bpm", F.when(F.col("avg_heart_rate_bpm").between(25, 250), F.col("avg_heart_rate_bpm")))
    .withColumn("max_heart_rate_bpm", F.when(F.col("max_heart_rate_bpm").between(25, 250), F.col("max_heart_rate_bpm")))
    .withColumn("avg_power_w", F.when(F.col("avg_power_w") <= 2000, F.col("avg_power_w")))
    .withColumn("activity_type", F.coalesce("activity_type", F.lit("other")))
    .withColumn("activity_family", F.coalesce("activity_family", F.lit("other")))
    .withColumn("is_indoor", F.col("is_trainer") | F.coalesce(F.col("is_indoor_default"), F.lit(False)))
    .withColumn("end_time_utc", (F.unix_timestamp("start_time_utc") + F.col("elapsed_duration_s")).cast("timestamp"))
    .withColumn("local_date", F.to_date("start_time_local"))
    .withColumn(
        "workout_intent",
        F.when(F.col("strava_workout_type").isin(1, 11), F.lit("race"))
         .when(F.col("strava_workout_type") == 2, F.lit("long"))
         .when(F.col("strava_workout_type").isin(3, 12), F.lit("workout"))
         .otherwise(F.lit("general")))
    .withColumn("equipment_id", F.when(F.col("source_equipment_id").isNotNull(),
                                        hash_key(F.col("source_system"), F.lit("equipment"), F.col("source_equipment_id"))))
    .cache()
)
print(f"Workouts parsed: {workouts_checked.count()}, valid: {workouts_valid.count()}")

In [ ]:
# Users, linked source accounts and equipment

ATHLETE_SCHEMA = (
    "id long, username string, firstname string, city string, state string, country string, sex string, premium boolean, "
    "summit boolean, created_at string, updated_at string, weight double, ftp double, measurement_preference string, "
    "bikes array<struct<id:string, primary:boolean, name:string, distance:double, retired:boolean>>, "
    "shoes array<struct<id:string, primary:boolean, name:string, distance:double, retired:boolean>>"
)
athlete_snapshots = (
    athlete_raw
    .select("ingested_at", "batch_id", "athlete_id", F.from_json("raw_payload", ATHLETE_SCHEMA).alias("p"))
    .withColumn("source_user_id", F.coalesce(F.col("p.id").cast("string"), F.col("athlete_id")))
    .where(F.col("source_user_id").isNotNull())
    .withColumn("user_id", strava_user_id(F.col("source_user_id")))
)
athlete_latest = latest(athlete_snapshots, ["source_user_id"])

tz_rank = Window.partitionBy("user_id").orderBy(F.desc("n"), F.desc("last_seen"))
primary_tz = (
    workouts_valid.where(F.col("timezone_name").isNotNull())
    .groupBy("user_id", "timezone_name").agg(F.count("*").alias("n"), F.max("start_time_utc").alias("last_seen"))
    .withColumn("_rn", F.row_number().over(tz_rank)).where("_rn = 1")
    .select("user_id", F.col("timezone_name").alias("primary_timezone_name"))
)
sync_window = (
    workouts_valid.groupBy("user_id", "source_user_id")
    .agg(F.min("ingested_at").alias("first_synced_at"), F.max("ingested_at").alias("last_synced_at"))
    .unionByName(athlete_snapshots.groupBy("user_id", "source_user_id")
                 .agg(F.min("ingested_at").alias("first_synced_at"), F.max("ingested_at").alias("last_synced_at")))
    .groupBy("user_id", "source_user_id")
    .agg(F.min("first_synced_at").alias("first_synced_at"), F.max("last_synced_at").alias("last_synced_at"))
)
profile = athlete_latest.select(
    "user_id", "source_user_id",
    F.col("p.firstname").alias("display_name"),
    F.when(F.col("p.sex") == "M", "male").when(F.col("p.sex") == "F", "female").alias("sex"),
    F.col("p.city").alias("home_city"), F.col("p.state").alias("home_region"), F.col("p.country").alias("home_country"),
    F.when(F.col("p.measurement_preference") == "feet", "imperial").otherwise("metric").alias("measurement_system"),
    F.col("p.username").alias("source_username"),
    F.to_timestamp("p.created_at").alias("source_account_created_at"),
    (F.coalesce(F.col("p.summit"), F.lit(False)) | F.coalesce(F.col("p.premium"), F.lit(False))).alias("is_premium"),
)
users = sync_window.join(profile, ["user_id", "source_user_id"], "left").join(primary_tz, "user_id", "left")

dim_user = users.select(
    "user_id", "display_name", "sex",
    F.lit(None).cast("date").alias("birth_date"),
    F.lit(None).cast("double").alias("height_cm"),
    "home_city", "home_region", "home_country",
    F.coalesce("measurement_system", F.lit("metric")).alias("measurement_system"),
    "primary_timezone_name",
    F.lit(SOURCE_SYSTEM).alias("primary_source_system"),
    F.col("first_synced_at").alias("created_at"),
    F.lit(PROCESSED_AT).cast("timestamp").alias("updated_at"),
)
user_source_account = users.select(
    hash_key(F.lit(SOURCE_SYSTEM), F.col("source_user_id")).alias("user_source_account_id"),
    "user_id",
    F.lit(SOURCE_SYSTEM).alias("source_system"),
    "source_user_id", "source_username", "source_account_created_at", "is_premium",
    F.lit(None).cast("string").alias("granted_scopes"),
    F.lit("active").alias("connection_status"),
    "first_synced_at", "last_synced_at",
)

gear_from_profile = (
    athlete_latest.select("user_id", F.explode("p.bikes").alias("g")).withColumn("equipment_type", F.lit("bike"))
    .unionByName(athlete_latest.select("user_id", F.explode("p.shoes").alias("g")).withColumn("equipment_type", F.lit("shoes")))
    .select("user_id", F.col("g.id").alias("source_equipment_id"), "equipment_type", F.col("g.name").alias("equipment_name"),
            F.col("g.distance").alias("total_distance_m"), F.coalesce(F.col("g.primary"), F.lit(False)).alias("is_primary"),
            F.coalesce(F.col("g.retired"), F.lit(False)).alias("is_retired"))
)
gear_usage = (
    workouts_valid.where(F.col("source_equipment_id").isNotNull())
    .groupBy("user_id", "source_equipment_id")
    .agg(F.max("equipment_name").alias("usage_name"), F.min("local_date").alias("first_used_date"),
         F.max("local_date").alias("last_used_date"), F.count("*").cast("int").alias("workout_count"),
         F.sum("distance_m").alias("tracked_distance_m"))
)
user_equipment = (
    gear_usage.join(gear_from_profile, ["user_id", "source_equipment_id"], "full")
    .select(
        hash_key(F.lit(SOURCE_SYSTEM), F.lit("equipment"), F.col("source_equipment_id")).alias("equipment_id"),
        "user_id",
        F.lit(SOURCE_SYSTEM).alias("source_system"),
        "source_equipment_id",
        F.coalesce("equipment_type",
                   F.when(F.col("source_equipment_id").startswith("b"), "bike")
                    .when(F.col("source_equipment_id").startswith("g"), "shoes")
                    .otherwise("unknown")).alias("equipment_type"),
        F.coalesce("equipment_name", "usage_name").alias("equipment_name"),
        F.coalesce("total_distance_m", "tracked_distance_m").alias("total_distance_m"),
        F.coalesce("is_primary", F.lit(False)).alias("is_primary"),
        F.coalesce("is_retired", F.lit(False)).alias("is_retired"),
        "first_used_date", "last_used_date",
        F.coalesce("workout_count", F.lit(0)).alias("workout_count"),
    )
)

write_table(dim_user, "UserProfile", scope=f"primary_source_system = '{SOURCE_SYSTEM}'")
write_table(user_source_account, "UserSourceAccount")
write_table(user_equipment, "UserEquipment")

In [ ]:
# Workout time series (one row per sample). GPS coordinates are intentionally not stored.

STREAM_SCHEMA = ", ".join(
    [f"{k} struct<data:array<double>>" for k in
     ["time", "distance", "heartrate", "watts", "cadence", "velocity_smooth", "altitude", "grade_smooth", "temp"]]
    + ["moving struct<data:array<boolean>>"]
)

streams = (
    streams_raw
    .select(F.col("source_record_id").alias("source_workout_id"), F.from_json("raw_payload", STREAM_SCHEMA).alias("s"))
    .select(
        "source_workout_id",
        F.col("s.time.data").alias("t"), F.col("s.distance.data").alias("d"), F.col("s.heartrate.data").alias("hr"),
        F.col("s.watts.data").alias("w"), F.col("s.cadence.data").alias("cad"), F.col("s.velocity_smooth.data").alias("v"),
        F.col("s.altitude.data").alias("alt"), F.col("s.grade_smooth.data").alias("g"), F.col("s.temp.data").alias("tmp"),
        F.col("s.moving.data").alias("mv"),
    )
    .where(F.col("t").isNotNull() & (F.size("t") > 0))
)

samples = streams.select(
    "source_workout_id",
    F.explode(F.expr(
        "transform(sequence(0, size(t) - 1), i -> named_struct("
        "'i', i, 't', t[i], 'd', d[i], 'hr', hr[i], 'w', w[i], 'cad', cad[i], 'v', v[i], "
        "'alt', alt[i], 'g', g[i], 'tmp', tmp[i], 'mv', mv[i]))"
    )).alias("x"),
)

sample_window = Window.partitionBy("workout_id").orderBy("sample_offset_s")
fact_workout_timeseries = (
    samples
    .join(workouts_valid.select("source_workout_id", "workout_id", "user_id", "start_time_utc"), "source_workout_id")
    .select(
        "workout_id", "user_id",
        F.lit(SOURCE_SYSTEM).alias("source_system"),
        F.col("x.i").cast("int").alias("sample_index"),
        F.col("x.t").cast("int").alias("sample_offset_s"),
        (F.unix_timestamp("start_time_utc") + F.col("x.t").cast("long")).cast("timestamp").alias("sample_time_utc"),
        F.round("x.d", 1).alias("distance_m"),
        F.when(F.col("x.hr").between(25, 250), F.col("x.hr")).cast("int").alias("heart_rate_bpm"),
        F.when(F.col("x.w").between(0, 2500), F.col("x.w")).cast("int").alias("power_w"),
        F.col("x.cad").cast("int").alias("cadence"),
        F.round("x.v", 3).alias("speed_mps"),
        F.round("x.alt", 1).alias("altitude_m"),
        F.round("x.g", 1).alias("grade_pct"),
        F.col("x.tmp").cast("int").alias("temperature_c"),
        F.col("x.mv").alias("is_moving"),
    )
    .withColumn(
        "sample_duration_s",
        F.least(F.coalesce(F.lead("sample_offset_s").over(sample_window) - F.col("sample_offset_s"), F.lit(1)), F.lit(10)).cast("int"),
    )
)

write_table(fact_workout_timeseries, "WorkoutSample")
timeseries = read_table("WorkoutSample").where(SOURCE_SCOPE)

In [ ]:
# Physiology: max heart rate, power best efforts, FTP and training zones

last_workout_date = workouts_valid.groupBy("user_id").agg(F.max("local_date").alias("last_date"))

max_hr = (
    workouts_valid.join(last_workout_date, "user_id")
    .where((F.col("local_date") >= F.date_sub("last_date", 365)) & ~F.array_contains("dq_warnings", "implausible_max_heart_rate"))
    .where(F.col("max_heart_rate_bpm").isNotNull())
    .groupBy("user_id")
    .agg(F.max("max_heart_rate_bpm").alias("max_hr"),
         F.max_by("start_time_utc", "max_heart_rate_bpm").alias("max_hr_observed_at"),
         F.max_by("workout_id", "max_heart_rate_bpm").alias("max_hr_workout_id"))
)

# Mean-maximal power for standard durations, computed from the time series (works for any power-capable source).
# Uses a running energy sum + row lag (linear time); assumes ~1 Hz samples and skips windows that span pauses (>10% longer).
POWER_DURATIONS = {5: "5 s", 60: "1 min", 300: "5 min", 600: "10 min", 900: "15 min", 1200: "20 min", 3600: "60 min"}
power_workouts = timeseries.groupBy("workout_id").agg(F.max("power_w").alias("mx")).where("mx > 0").select("workout_id")
ordered = Window.partitionBy("workout_id").orderBy("sample_offset_s")
power_samples = (
    timeseries.join(power_workouts, "workout_id")
    .select("workout_id", "user_id", "sample_offset_s",
            (F.coalesce("power_w", F.lit(0)) * F.col("sample_duration_s")).cast("double").alias("energy_j"))
    .withColumn("energy_before_j", F.sum("energy_j").over(ordered.rowsBetween(Window.unboundedPreceding, Window.currentRow)) - F.col("energy_j"))
)
window_cols = []
for seconds in POWER_DURATIONS:
    span = F.col("sample_offset_s") - F.lag("sample_offset_s", seconds).over(ordered)
    energy = F.col("energy_before_j") - F.lag("energy_before_j", seconds).over(ordered)
    window_cols.append(F.when(span.between(seconds, seconds * 1.1), energy / span).alias(f"p_{seconds}"))
mean_max_power = (
    power_samples.select("workout_id", "user_id", *window_cols)
    .groupBy("workout_id", "user_id")
    .agg(*[F.max(f"p_{s}").alias(f"p_{s}") for s in POWER_DURATIONS])
)
power_efforts = (
    mean_max_power
    .select("workout_id", "user_id", F.explode(F.array(*[
        F.struct(F.lit(s).alias("duration_s"), F.lit(name).alias("effort_name"), F.col(f"p_{s}").alias("avg_power_w"))
        for s, name in POWER_DURATIONS.items()])).alias("e"))
    .where(F.col("e.avg_power_w").isNotNull())
    .select("workout_id", "user_id", "e.*")
)

ftp_profile = (
    athlete_latest.where(F.col("p.ftp") > 0)
    .select("user_id", F.col("p.ftp").alias("ftp_w"), F.lit("strava_profile").alias("ftp_method"))
)
ftp_estimated = (
    power_efforts.where((F.col("duration_s") == 1200) & (F.col("avg_power_w") < 2000))
    .join(workouts_valid.where(F.col("activity_family") == "cycling").select("workout_id", "local_date", "start_time_utc"), "workout_id")
    .join(last_workout_date, "user_id")
    .where(F.col("local_date") >= F.date_sub("last_date", 90))
    .groupBy("user_id")
    .agg(F.round(F.max("avg_power_w") * 0.95, 0).alias("ftp_w"),
         F.max_by("start_time_utc", "avg_power_w").alias("ftp_observed_at"),
         F.max_by("workout_id", "avg_power_w").alias("ftp_workout_id"))
    .withColumn("ftp_method", F.lit("estimated_95pct_best_20min_power_90d"))
)
current_ftp = ftp_profile.unionByName(
    ftp_estimated.select("user_id", "ftp_w", "ftp_method").join(ftp_profile.select("user_id"), "user_id", "left_anti"))

HR_ZONE_NAMES = ["Recovery", "Endurance", "Tempo", "Threshold", "Anaerobic"]
POWER_ZONE_NAMES = ["Active recovery", "Endurance", "Tempo", "Threshold", "VO2max", "Anaerobic", "Neuromuscular"]
ZONES_SCHEMA = "heart_rate struct<custom_zones:boolean, zones:array<struct<min:double, max:double>>>, power struct<zones:array<struct<min:double, max:double>>>"
zones_parsed = zones_raw.select(
    strava_user_id(F.col("athlete_id")).alias("user_id"), F.to_date("ingested_at").alias("effective_from"),
    F.from_json("raw_payload", ZONES_SCHEMA).alias("z"))

source_hr_zones = zones_parsed.select(
    "user_id", "effective_from",
    F.when(F.col("z.heart_rate.custom_zones"), "strava_custom").otherwise("strava_default").alias("zone_model"),
    F.posexplode("z.heart_rate.zones").alias("pos", "zone"),
).select("user_id", F.lit("heart_rate").alias("zone_type"), "zone_model", (F.col("pos") + 1).alias("zone_number"),
         F.col("zone.min").alias("lower_bound"), F.when(F.col("zone.max") > 0, F.col("zone.max")).alias("upper_bound"),
         F.lit(None).cast("double").alias("reference_value"), "effective_from")

source_power_zones = zones_parsed.select(
    "user_id", "effective_from", F.posexplode("z.power.zones").alias("pos", "zone"),
).select("user_id", F.lit("power").alias("zone_type"), F.lit("strava_profile").alias("zone_model"),
         (F.col("pos") + 1).alias("zone_number"), F.col("zone.min").alias("lower_bound"),
         F.when(F.col("zone.max") > 0, F.col("zone.max")).alias("upper_bound"),
         F.lit(None).cast("double").alias("reference_value"), "effective_from")

hr_pct = spark.createDataFrame([(1, 0.0, 0.6), (2, 0.6, 0.7), (3, 0.7, 0.8), (4, 0.8, 0.9), (5, 0.9, None)],
                               "zone_number int, lo double, hi double")
derived_hr_zones = max_hr.crossJoin(hr_pct).select(
    "user_id", F.lit("heart_rate").alias("zone_type"), F.lit("derived_pct_max_hr").alias("zone_model"), "zone_number",
    F.round(F.col("lo") * F.col("max_hr")).alias("lower_bound"), F.round(F.col("hi") * F.col("max_hr")).alias("upper_bound"),
    F.col("max_hr").alias("reference_value"), F.lit(PROCESSED_AT.date()).alias("effective_from"))

coggan_pct = spark.createDataFrame(
    [(1, 0.0, 0.55), (2, 0.55, 0.75), (3, 0.75, 0.90), (4, 0.90, 1.05), (5, 1.05, 1.20), (6, 1.20, 1.50), (7, 1.50, None)],
    "zone_number int, lo double, hi double")
derived_power_zones = current_ftp.crossJoin(coggan_pct).select(
    "user_id", F.lit("power").alias("zone_type"), F.concat(F.lit("coggan_"), F.col("ftp_method")).alias("zone_model"), "zone_number",
    F.round(F.col("lo") * F.col("ftp_w")).alias("lower_bound"), F.round(F.col("hi") * F.col("ftp_w")).alias("upper_bound"),
    F.col("ftp_w").alias("reference_value"), F.lit(PROCESSED_AT.date()).alias("effective_from"))

hr_zones = source_hr_zones.unionByName(derived_hr_zones.join(source_hr_zones.select("user_id").distinct(), "user_id", "left_anti"))
power_zones = source_power_zones.unionByName(derived_power_zones.join(source_power_zones.select("user_id").distinct(), "user_id", "left_anti"))

user_physiology_zone = (
    hr_zones.unionByName(power_zones)
    .withColumn("zone_name", F.when(F.col("zone_type") == "heart_rate", F.element_at(F.array(*[F.lit(n) for n in HR_ZONE_NAMES]), F.col("zone_number")))
                              .otherwise(F.element_at(F.array(*[F.lit(n) for n in POWER_ZONE_NAMES]), F.col("zone_number"))))
    .select(
        hash_key(F.col("user_id"), F.col("zone_type"), F.col("zone_model"), F.col("zone_number"), F.col("effective_from")).alias("zone_id"),
        "user_id",
        F.lit(SOURCE_SYSTEM).alias("source_system"),
        "zone_type", "zone_model", "zone_number", "zone_name", "lower_bound", "upper_bound",
        F.when(F.col("zone_type") == "heart_rate", "bpm").otherwise("W").alias("unit"),
        "reference_value", "effective_from",
        F.lit(True).alias("is_current"),
    )
)
write_table(user_physiology_zone, "UserTrainingZone")
zones = read_table("UserTrainingZone").where(SOURCE_SCOPE).where("is_current")

# Time in zone per workout (moving samples only)
moving = timeseries.where(F.col("is_moving").isNull() | F.col("is_moving"))
hr_time = (
    moving.where(F.col("heart_rate_bpm").isNotNull()).select("workout_id", "user_id", F.col("heart_rate_bpm").alias("value"), "sample_duration_s")
    .join(F.broadcast(zones.where("zone_type = 'heart_rate'")), "user_id")
)
power_time = (
    moving.where(F.col("power_w").isNotNull()).select("workout_id", "user_id", F.col("power_w").alias("value"), "sample_duration_s")
    .join(F.broadcast(zones.where("zone_type = 'power'")), "user_id")
)
zone_totals = Window.partitionBy("workout_id", "zone_type")
fact_workout_zone_time = (
    hr_time.unionByName(power_time)
    .where((F.col("value") >= F.col("lower_bound")) & (F.col("upper_bound").isNull() | (F.col("value") < F.col("upper_bound"))))
    .groupBy("workout_id", "user_id", "zone_type", "zone_model", "zone_number", "zone_name")
    .agg(F.sum("sample_duration_s").cast("long").alias("seconds_in_zone"))
    .withColumn("pct_of_zone_time", F.round(F.col("seconds_in_zone") / F.sum("seconds_in_zone").over(zone_totals) * 100, 1))
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
)
write_table(fact_workout_zone_time, "WorkoutZoneTime")

In [ ]:
# Workout segments (laps + 1 km splits) and best efforts (source distance PRs + derived power curve)

wk_keys = workouts_valid.select("workout_id", "user_id", "start_time_utc", "start_time_local", "laps", "splits_metric", "best_efforts")

laps = wk_keys.select("workout_id", "user_id", "start_time_utc", F.explode("laps").alias("s")).select(
    "workout_id", "user_id",
    F.lit("lap").alias("segment_type"),
    F.col("s.lap_index").alias("segment_index"),
    F.col("s.name").alias("segment_name"),
    F.to_timestamp("s.start_date").alias("start_time_utc"),
    (F.unix_timestamp(F.to_timestamp("s.start_date")) - F.unix_timestamp("start_time_utc")).cast("int").alias("start_offset_s"),
    F.col("s.elapsed_time").alias("elapsed_duration_s"),
    F.col("s.moving_time").alias("moving_duration_s"),
    F.col("s.distance").alias("distance_m"),
    F.col("s.total_elevation_gain").alias("elevation_gain_m"),
    F.lit(None).cast("double").alias("elevation_change_m"),
    F.col("s.average_speed").alias("avg_speed_mps"),
    F.col("s.max_speed").alias("max_speed_mps"),
    F.lit(None).cast("double").alias("grade_adjusted_speed_mps"),
    F.col("s.average_heartrate").alias("avg_heart_rate_bpm"),
    F.col("s.max_heartrate").alias("max_heart_rate_bpm"),
    F.col("s.average_watts").alias("avg_power_w"),
    F.col("s.average_cadence").alias("avg_cadence"),
    F.col("s.pace_zone").alias("pace_zone"),
)
splits = wk_keys.select("workout_id", "user_id", F.explode("splits_metric").alias("s")).select(
    "workout_id", "user_id",
    F.lit("split_1km").alias("segment_type"),
    F.col("s.split").alias("segment_index"),
    F.concat(F.lit("km "), F.col("s.split")).alias("segment_name"),
    F.lit(None).cast("timestamp").alias("start_time_utc"),
    F.lit(None).cast("int").alias("start_offset_s"),
    F.col("s.elapsed_time").alias("elapsed_duration_s"),
    F.col("s.moving_time").alias("moving_duration_s"),
    F.col("s.distance").alias("distance_m"),
    F.lit(None).cast("double").alias("elevation_gain_m"),
    F.col("s.elevation_difference").alias("elevation_change_m"),
    F.col("s.average_speed").alias("avg_speed_mps"),
    F.lit(None).cast("double").alias("max_speed_mps"),
    F.col("s.average_grade_adjusted_speed").alias("grade_adjusted_speed_mps"),
    F.col("s.average_heartrate").alias("avg_heart_rate_bpm"),
    F.lit(None).cast("double").alias("max_heart_rate_bpm"),
    F.lit(None).cast("double").alias("avg_power_w"),
    F.lit(None).cast("double").alias("avg_cadence"),
    F.col("s.pace_zone").alias("pace_zone"),
)
fact_workout_segment = (
    laps.unionByName(splits)
    .withColumn("segment_id", hash_key(F.col("workout_id"), F.col("segment_type"), F.col("segment_index")))
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
)
write_table(fact_workout_segment, "WorkoutSegment")

distance_efforts = wk_keys.select("workout_id", "user_id", "start_time_local", F.explode("best_efforts").alias("e")).select(
    "workout_id", "user_id",
    F.lit("distance").alias("effort_type"),
    F.col("e.name").alias("effort_name"),
    F.col("e.distance").alias("distance_m"),
    F.col("e.elapsed_time").cast("int").alias("duration_s"),
    F.lit(None).cast("double").alias("avg_power_w"),
    F.round(F.col("e.distance") / F.col("e.elapsed_time"), 3).alias("avg_speed_mps"),
    F.coalesce(F.to_timestamp("e.start_date"), F.lit(None).cast("timestamp")).alias("start_time_utc"),
    F.col("e.pr_rank").alias("source_pr_rank"),
    F.lit("source").alias("effort_origin"),
)
derived_power_efforts = (
    power_efforts.join(workouts_valid.select("workout_id", "start_time_utc"), "workout_id")
    .select(
        "workout_id", "user_id",
        F.lit("power_duration").alias("effort_type"),
        F.concat(F.col("effort_name"), F.lit(" power")).alias("effort_name"),
        F.lit(None).cast("double").alias("distance_m"),
        F.col("duration_s").cast("int").alias("duration_s"),
        F.round("avg_power_w", 1).alias("avg_power_w"),
        F.lit(None).cast("double").alias("avg_speed_mps"),
        "start_time_utc",
        F.lit(None).cast("int").alias("source_pr_rank"),
        F.lit("derived_timeseries").alias("effort_origin"),
    )
)
fact_workout_best_effort = (
    distance_efforts.unionByName(derived_power_efforts)
    .withColumn("best_effort_id", hash_key(F.col("workout_id"), F.col("effort_type"), F.col("effort_name")))
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
)
write_table(fact_workout_best_effort, "WorkoutBestEffort")

In [ ]:
# Workout fact with standardized training-load metrics and cross-source duplicate detection

trimp = (
    read_table("WorkoutZoneTime").where(SOURCE_SCOPE).where("zone_type = 'heart_rate'")
    .groupBy("workout_id").agg(F.round(F.sum(F.col("seconds_in_zone") / 60.0 * F.col("zone_number")), 1).alias("trimp_edwards"))
)
ts_flags = timeseries.groupBy("workout_id").agg(
    F.lit(True).alias("has_timeseries"),
    F.max(F.col("power_w").isNotNull().cast("int")).cast("boolean").alias("has_power_samples"),
)
priority = ref_data_source.select("source_system", "workout_priority")
TRIMP_PER_TSS = 2.4

workouts_enriched = (
    workouts_valid
    .join(current_ftp.select("user_id", "ftp_w"), "user_id", "left")
    .join(max_hr.select("user_id", "max_hr"), "user_id", "left")
    .join(trimp, "workout_id", "left")
    .join(ts_flags, "workout_id", "left")
    .join(F.broadcast(priority), "source_system", "left")
    .withColumn("has_timeseries", F.coalesce("has_timeseries", F.lit(False)))
    .withColumn("has_power", (F.col("avg_power_w") > 0) | F.coalesce("has_power_samples", F.lit(False)))
    .withColumn("intensity_factor", F.when((F.col("normalized_power_w") > 0) & (F.col("ftp_w") > 0),
                                           F.round(F.col("normalized_power_w") / F.col("ftp_w"), 3)))
    .withColumn("tss", F.when(F.col("intensity_factor").isNotNull(),
                              F.round(F.col("moving_duration_s") * F.col("normalized_power_w") * F.col("intensity_factor")
                                      / (F.col("ftp_w") * 3600) * 100, 1)))
    .withColumn("hr_intensity_ratio", F.when(F.col("max_hr") > 0, F.round(F.col("avg_heart_rate_bpm") / F.col("max_hr"), 3)))
    # One load scale for all sports (TSS units: one hour at threshold = 100). Edwards TRIMP weights threshold
    # (80-90 % of max HR) with 4 per minute = 240 per hour, so heart-rate load is divided by 2.4.
    .withColumn("training_load_score", F.coalesce("tss", F.round(F.col("trimp_edwards") / TRIMP_PER_TSS, 1), "source_effort_score"))
    .withColumn("training_load_method",
                F.when(F.col("tss").isNotNull(), "tss_power")
                 .when(F.col("trimp_edwards").isNotNull(), "hr_trimp_tss_scaled")
                 .when(F.col("source_effort_score").isNotNull(), "strava_relative_effort"))
    .withColumn("dedupe_group_id", hash_key(F.col("user_id"), F.col("activity_family"),
                                            F.floor(F.unix_timestamp("start_time_utc") / 600)))
)
dedupe_rank = Window.partitionBy("dedupe_group_id").orderBy(
    F.coalesce("workout_priority", F.lit(99)), F.desc("has_timeseries"), F.desc("has_detail"), F.desc("ingested_at"))

fact_workout = (
    workouts_enriched
    .withColumn("is_primary_record", F.row_number().over(dedupe_rank) == 1)
    .select(
        "workout_id", "user_id", "source_system", "source_workout_id", "source_external_id",
        "activity_type", "activity_family", "source_activity_type", "workout_intent",
        "workout_name", "workout_description",
        "start_time_utc", "end_time_utc", "start_time_local", "local_date", "timezone_name", "utc_offset_s",
        "elapsed_duration_s", "moving_duration_s", "distance_m",
        "elevation_gain_m", "elevation_high_m", "elevation_low_m",
        "avg_speed_mps", "max_speed_mps",
        "avg_heart_rate_bpm", "max_heart_rate_bpm",
        "avg_power_w", "max_power_w", "normalized_power_w", "is_power_from_device",
        "avg_cadence", "avg_temperature_c",
        "energy_kj", "active_energy_kcal",
        "perceived_exertion", "source_effort_score",
        F.col("ftp_w").alias("ftp_at_calculation_w"), "intensity_factor", "tss", "trimp_edwards", "hr_intensity_ratio",
        "training_load_score", "training_load_method",
        "is_indoor", "is_trainer", "is_commute", "is_manual", "is_private",
        "has_heart_rate", "has_power", "has_timeseries", "has_detail",
        "device_name", "equipment_id",
        "dedupe_group_id", "is_primary_record",
        F.concat_ws(",", "dq_warnings").alias("data_quality_flags"),
        "source_attributes_json",
        "ingested_at", "batch_id",
    )
)
write_table(fact_workout, "Workout")

In [ ]:
# Health observations (long format): profile weight/FTP history from Strava + derived physiology

def observation(df, metric, unit, method, value_col, observed_col, record_col, ingested_col="ingested_at"):
    return df.select(
        "user_id",
        F.lit(SOURCE_SYSTEM).alias("source_system"),
        F.col(record_col).cast("string").alias("source_record_id"),
        F.lit(metric).alias("metric_code"),
        F.col(value_col).cast("double").alias("value"),
        F.lit(unit).alias("unit"),
        F.col(observed_col).cast("timestamp").alias("observed_at_utc"),
        F.lit(None).cast("timestamp").alias("period_start_utc"),
        F.lit(None).cast("timestamp").alias("period_end_utc"),
        F.lit(method).alias("observation_method"),
        F.lit(None).cast("string").alias("device_name"),
        F.col(ingested_col).alias("ingested_at"),
    )

profile_history = athlete_snapshots.withColumn("profile_updated_at", F.coalesce(F.to_timestamp("p.updated_at"), F.col("ingested_at")))
weight_obs = observation(profile_history.where(F.col("p.weight") > 0).withColumn("w", F.col("p.weight")),
                         "body_weight_kg", "kg", "self_reported_profile", "w", "profile_updated_at", "source_user_id")
ftp_obs = observation(profile_history.where(F.col("p.ftp") > 0).withColumn("f", F.col("p.ftp")),
                      "ftp_watts", "W", "self_reported_profile", "f", "profile_updated_at", "source_user_id")
ftp_est_obs = observation(ftp_estimated.withColumn("ingested_at", F.lit(PROCESSED_AT).cast("timestamp")),
                          "ftp_watts", "W", "estimated_95pct_best_20min_power_90d", "ftp_w", "ftp_observed_at", "ftp_workout_id")
max_hr_obs = observation(max_hr.withColumn("ingested_at", F.lit(PROCESSED_AT).cast("timestamp")),
                         "max_heart_rate_bpm", "bpm", "max_observed_workout_365d", "max_hr", "max_hr_observed_at", "max_hr_workout_id")

fact_health_observation = (
    weight_obs.unionByName(ftp_obs).unionByName(ftp_est_obs).unionByName(max_hr_obs)
    .withColumn("local_date", F.to_date("observed_at_utc"))
    .withColumn("observation_id", hash_key(F.col("user_id"), F.col("source_system"), F.col("metric_code"),
                                           F.col("observation_method"), F.col("observed_at_utc")))
    .dropDuplicates(["observation_id"])
)
write_table(fact_health_observation, "HealthObservation")

In [ ]:
# Data quality issues: rejected/flagged workouts and failed API calls

workout_issues = (
    workouts_checked
    .select("source_workout_id", "batch_id",
            F.explode(F.concat(
                F.transform("dq_errors", lambda r: F.struct(r.alias("rule_code"), F.lit("error").alias("severity"))),
                F.transform("dq_warnings", lambda r: F.struct(r.alias("rule_code"), F.lit("warning").alias("severity"))),
            )).alias("i"))
    .select(F.lit("workout").alias("entity_name"), F.col("source_workout_id").alias("source_record_id"),
            "i.rule_code", "i.severity", F.lit(None).cast("string").alias("issue_detail"), "batch_id")
)
api_issues = (
    latest(api_raw, ["source_entity", "source_record_id"])
    .where("http_status <> 200")
    .select(F.col("source_entity").alias("entity_name"), "source_record_id",
            F.concat(F.lit("api_http_"), F.col("http_status").cast("string")).alias("rule_code"),
            F.when(F.col("http_status") == 404, "warning").otherwise("error").alias("severity"),
            F.substring("raw_payload", 1, 500).alias("issue_detail"), "batch_id")
)
dq_issue = (
    workout_issues.unionByName(api_issues)
    .withColumn("source_system", F.lit(SOURCE_SYSTEM))
    .withColumn("dq_issue_id", hash_key(F.col("source_system"), F.col("entity_name"), F.col("source_record_id"), F.col("rule_code")))
    .withColumn("detected_at", F.lit(PROCESSED_AT).cast("timestamp"))
)
write_table(dq_issue, "DataQualityIssue")

In [ ]:
# Contract tables (sleep, nutrition, goals, preferences) are created by the Silver_Reference_Data notebook.

In [ ]:
for table_name in ["Workout", "WorkoutSample", "WorkoutZoneTime", "WorkoutSegment"]:
    spark.sql(f"OPTIMIZE delta.`Tables/{table_name}`")

workouts_valid.unpersist()
result = {"status": "success", "layer": "silver", "source_system": SOURCE_SYSTEM, "processed_at": PROCESSED_AT.isoformat(), "row_counts": row_counts}
print(json.dumps(result, indent=2))
notebookutils.notebook.exit(json.dumps(result))